# Concrete Compressive Strength
### From mix proportions to strength: physics, machine learning and mix design

---

**Background.** Compressive strength is the property that governs the structural design of concrete. It depends on how the mix is proportioned and on how long the concrete has cured. The dataset (Yeh, 1998; UCI Machine Learning Repository) records 1,030 laboratory tests with:
- seven mix components in kg per m³: cement, blast-furnace slag, fly ash, water, superplasticizer, coarse aggregate and fine aggregate;
- the age at testing in days;
- the measured compressive strength in MPa.

**What this study adds to the original analysis**
1. **Engineering ratios.** Classic concrete technology holds that strength is governed by *ratios*, above all the water-to-binder ratio (Abrams' law, 1918), together with a logarithmic gain of strength with age. The notebook tests this against the data and uses it as an interpretable physical benchmark.
2. **Honest validation.** The same concrete mix is usually tested at several ages. When rows are split at random, the model has already seen the test mix at other ages. Here all validation keeps each mix entirely on one side of the split, and the notebook measures how much the random split overstates performance.
3. **Seventeen models compared fairly,** with tuning, confidence intervals and prediction intervals.
4. **From prediction to decision.** Strength-gain curves for any mix, and an optimiser that designs the cheapest or lowest-CO₂ mix that reaches a target 28-day strength while staying within the range of the data.
5. **Learnings.** Every section ends with a "What we learn" box computed from the run, and §13 gathers them all.

| § | Content |
|---|---|
| 1 | Setup |
| 2 | Data acquisition and harmonisation |
| 3 | Data audit: duplicates, repeated mixes, mass balance |
| 4 | Concrete technology in the data: Abrams' law, strength gain, supplementary materials |
| 5 | Feature engineering |
| 6 | Validation design: random vs mix-grouped |
| 7 | Model benchmark: physics model + 16 ML models |
| 8 | Hyperparameter tuning |
| 9 | Hold-out evaluation and prediction intervals |
| 10 | Interpretation |
| 11 | Strength-gain curves |
| 12 | Mix design optimiser |
| 13 | Key learnings |

## 1 · Setup

In [ ]:
%pip install -q xgboost lightgbm catboost xlrd ucimlrepo

In [ ]:
import io
import sys
import json
import time
import shutil
import platform
import warnings
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats, optimize
from scipy.stats import loguniform, randint, uniform
from IPython.display import display, Markdown

import sklearn
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import KFold, GroupShuffleSplit, cross_validate, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.neural_network import MLPRegressor

import xgboost
import lightgbm
import catboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
VERSIONS = {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__,
            "lightgbm": lightgbm.__version__, "catboost": catboost.__version__}
print(" | ".join(f"{k} {v}" for k, v in VERSIONS.items()))

In [ ]:
CONFIG = {
    "seed": 42,
    "test_fraction": 0.20,
    "cv_splits": 5,
    "cv_repeats": 3,
    "search_iter": 20,
    "bootstrap": 1000,
    "conformal_alpha": 0.10,
    "design_targets_mpa": [25, 30, 40, 50, 60],
    "de_maxiter": 120,
    "quick_run": False,
    "uci_url": "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls",
    "price_per_kg": {"cement": 0.12, "slag": 0.06, "fly_ash": 0.04, "water": 0.001, "superplasticizer": 2.50,
                     "coarse_agg": 0.020, "fine_agg": 0.018},
    "co2_per_kg": {"cement": 0.90, "slag": 0.07, "fly_ash": 0.03, "water": 0.0003, "superplasticizer": 1.90,
                   "coarse_agg": 0.005, "fine_agg": 0.004},
    "density_kg_m3": {"cement": 3150, "slag": 2900, "fly_ash": 2300, "water": 1000, "superplasticizer": 1200,
                      "coarse_agg": 2700, "fine_agg": 2650},
    "air_content": 0.02,
}
if CONFIG["quick_run"]:
    CONFIG.update(cv_repeats=1, search_iter=3, bootstrap=200, de_maxiter=20)

HOME = (Path("/content") if IN_COLAB else Path.cwd()) / "concrete_study"
DIRS = {k: HOME / k for k in ("figures", "tables", "model")}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
SEED = CONFIG["seed"]

The prices and CO₂ factors in `CONFIG` are **illustrative** values of typical magnitude. They only affect the mix design optimiser in §12 and should be replaced with local values before any real use.

In [ ]:
CEMENT_GREY, INK, ORANGE, BLUE, GREEN, SAND = "#6B7280", "#111827", "#EA580C", "#0284C7", "#65A30D", "#E7E5E4"
AGE_COLORS = {"early (≤7 d)": "#F59E0B", "standard (14–28 d)": "#0284C7", "late (>28 d)": "#7C3AED"}

sns.set_theme(style="white", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "axes.titleweight": "bold", "axes.titlesize": 11.5, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#F1F5F9",
    "axes.edgecolor": "#9CA3AF", "legend.frameon": False,
    "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, GREEN, "#7C3AED", CEMENT_GREY, "#BE123C"]),
})

LEARNINGS = defaultdict(list)


def learn(category, text):
    LEARNINGS[category].append(text)
    display(Markdown(f"> **What we learn · {category}.** {text}"))


def savefig(fig, name):
    fig.savefig(DIRS["figures"] / f"{name}.png", bbox_inches="tight", facecolor="white")


def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, p)))

## 2 · Data Acquisition and Harmonisation
Sources are tried in this order:
1. a local copy of `Concrete_Data.xls` or a CSV;
2. the UCI repository file;
3. the `ucimlrepo` package (dataset 165);
4. a file-upload prompt.

The very long original column names are mapped to short names by keyword.

In [ ]:
COLUMN_KEYS = [("cement", "cement"), ("slag", "slag"), ("fly", "fly_ash"), ("water", "water"), ("superplastic", "superplasticizer"),
               ("coarse", "coarse_agg"), ("fine", "fine_agg"), ("age", "age"), ("strength", "strength")]


def harmonise(frame):
    mapping = {}
    for col in frame.columns:
        lc = str(col).lower()
        for key, name in COLUMN_KEYS:
            if key in lc and name not in mapping.values():
                mapping[col] = name
                break
    out = frame.rename(columns=mapping)
    missing = [n for _, n in COLUMN_KEYS if n not in out.columns]
    if missing:
        raise ValueError(f"Could not identify columns: {missing}")
    return out[[n for _, n in COLUMN_KEYS]].apply(pd.to_numeric, errors="coerce")


def acquire():
    for p in [Path("Concrete_Data.xls"), Path("data/Concrete_Data.xls"), Path("/content/Concrete_Data.xls")]:
        if p.exists():
            return pd.read_excel(p), f"local file {p}"
    for p in [Path("concrete_data.csv"), Path("Concrete_Data.csv"), Path("data/concrete_data.csv"), Path("/content/concrete_data.csv")]:
        if p.exists():
            return pd.read_csv(p), f"local file {p}"
    try:
        return pd.read_excel(CONFIG["uci_url"]), "UCI repository (xls)"
    except Exception as exc:
        print(f"UCI file unavailable: {type(exc).__name__}")
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=165)
        return pd.concat([ds.data.features, ds.data.targets], axis=1), "ucimlrepo"
    except Exception as exc:
        print(f"ucimlrepo unavailable: {type(exc).__name__}")
    if IN_COLAB:
        from google.colab import files
        up = files.upload()
        key = next(iter(up))
        reader = pd.read_excel if key.endswith((".xls", ".xlsx")) else pd.read_csv
        return reader(io.BytesIO(up[key])), f"upload {key}"
    raise FileNotFoundError("Concrete data not found")


raw, source = acquire()
concrete = harmonise(raw)
INGREDIENTS = ["cement", "slag", "fly_ash", "water", "superplasticizer", "coarse_agg", "fine_agg"]
RAW_INPUTS = INGREDIENTS + ["age"]
print(f"Loaded {len(concrete)} tests from {source}")
display(concrete.describe().T.style.format(precision=1))

## 3 · Data Audit

### 3.1 Duplicates and repeated mixes
The audit counts exact duplicate rows, which are the same test recorded twice. It also identifies each distinct **mix** by its seven proportions and counts how many ages each mix was tested at. Repeated mixes are the reason the validation in §6 has to be grouped by mix.

In [ ]:
dup_rows = int(concrete.duplicated().sum())
concrete["mix_id"] = concrete[INGREDIENTS].round(1).astype(str).agg("|".join, axis=1).factorize()[0]
ages_per_mix = concrete.groupby("mix_id")["age"].nunique()
audit = pd.Series({
    "tests": len(concrete),
    "exact duplicate rows": dup_rows,
    "distinct mixes": concrete["mix_id"].nunique(),
    "mixes tested at ≥2 ages": int((ages_per_mix >= 2).sum()),
    "tests belonging to multi-age mixes": int(concrete["mix_id"].map(ages_per_mix).ge(2).sum()),
    "distinct test ages": concrete["age"].nunique(),
    "missing values": int(concrete.isna().sum().sum()),
}, name="value")
display(audit.to_frame())

concrete["age_group"] = pd.cut(concrete["age"], [0, 7, 28, 10_000], labels=list(AGE_COLORS))
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
age_counts = concrete["age"].value_counts().sort_index()
axes[0].bar(age_counts.index.astype(str), age_counts.values, color=CEMENT_GREY)
axes[0].set(title="Tests per age", xlabel="age [days]", ylabel="tests")
axes[0].tick_params(axis="x", rotation=45)
axes[1].hist(ages_per_mix, bins=np.arange(0.5, ages_per_mix.max() + 1.5), color=ORANGE, rwidth=0.85)
axes[1].set(title="Number of test ages per mix", xlabel="ages", ylabel="mixes")
fig.tight_layout()
savefig(fig, "3_1_structure")
plt.show()

learn("data", f"The {len(concrete)} tests contain {dup_rows} exact duplicate rows and only {concrete['mix_id'].nunique()} distinct mixes. "
      f"{int((ages_per_mix >= 2).sum())} mixes were tested at several ages. The effective number of independent designs "
      f"is therefore much smaller than the row count suggests.")

### 3.2 Mass balance and plausibility
The mass of all components in one m³ of normal concrete is typically 2,200–2,600 kg. Using the component densities in `CONFIG` and 2 % entrapped air, the absolute volume of each recorded mix should add up to about 1 m³. Mixes far outside these ranges point to recording errors.

In [ ]:
dens = CONFIG["density_kg_m3"]
concrete["total_mass"] = concrete[INGREDIENTS].sum(axis=1)
concrete["abs_volume"] = sum(concrete[c] / dens[c] for c in INGREDIENTS) + CONFIG["air_content"]
flags = concrete[(~concrete["total_mass"].between(2150, 2650)) | (~concrete["abs_volume"].between(0.95, 1.05))]

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
sns.histplot(concrete["total_mass"], bins=40, ax=axes[0], color=CEMENT_GREY)
axes[0].axvspan(2150, 2650, color=GREEN, alpha=0.1)
axes[0].set(title="Total mass per m³", xlabel="kg")
sns.histplot(concrete["abs_volume"], bins=40, ax=axes[1], color=BLUE)
axes[1].axvspan(0.95, 1.05, color=GREEN, alpha=0.1)
axes[1].set(title="Absolute volume incl. 2% air", xlabel="m³")
fig.tight_layout()
savefig(fig, "3_2_mass_balance")
plt.show()
print(f"Mixes outside plausibility bands: {flags['mix_id'].nunique()} ({len(flags)} tests)")
learn("data", f"The median mix totals {concrete['total_mass'].median():,.0f} kg and {concrete['abs_volume'].median():.3f} m³ per m³ of concrete. "
      f"{flags['mix_id'].nunique()} mixes fall outside the plausibility bands. "
      + ("The data are physically consistent overall." if flags["mix_id"].nunique() < 0.05 * concrete["mix_id"].nunique()
         else "A notable share of mixes looks inconsistent and should be reviewed."))

## 4 · Concrete Technology in the Data

### 4.1 Abrams' law
In 1918 Abrams stated that the strength of fully compacted concrete depends mainly on the ratio of water to cementitious material: $f_c = A / B^{\,w/b}$. Taking logarithms gives a straight line, $\ln f_c = \ln A - (w/b)\ln B$. This relationship is tested on the 28-day tests. The comparison is between how much of the variation in strength the water-to-cement ratio, the water-to-binder ratio, cement content alone and water content alone each explain.

In [ ]:
def add_ratios(df):
    out = df.copy()
    out["binder"] = out["cement"] + out["slag"] + out["fly_ash"]
    out["w_c"] = out["water"] / out["cement"]
    out["w_b"] = out["water"] / out["binder"]
    out["scm_frac"] = (out["slag"] + out["fly_ash"]) / out["binder"]
    return out


concrete = add_ratios(concrete)
d28 = concrete[concrete["age"] == 28]
r2_single = {}
for col in ["w_b", "w_c", "cement", "water"]:
    b, a = np.polyfit(d28[col], np.log(d28["strength"]), 1)
    r2_single[col] = r2_score(np.log(d28["strength"]), a + b * d28[col])
b_wb, a_wb = np.polyfit(d28["w_b"], np.log(d28["strength"]), 1)
A_abrams, B_abrams = np.exp(a_wb), np.exp(-b_wb)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sc = axes[0].scatter(d28["w_b"], d28["strength"], c=d28["scm_frac"], cmap="viridis", s=26, edgecolor="k", lw=0.2)
xs = np.linspace(d28["w_b"].min(), d28["w_b"].max(), 100)
axes[0].plot(xs, A_abrams / B_abrams ** xs, color=ORANGE, lw=2.5, label=f"Abrams fit: {A_abrams:.0f} / {B_abrams:.1f}^(w/b)")
axes[0].set(title="28-day strength vs water/binder ratio", xlabel="w/b", ylabel="strength [MPa]")
axes[0].legend()
fig.colorbar(sc, ax=axes[0], label="SCM fraction of binder")
bars = pd.Series(r2_single).sort_values()
axes[1].barh(bars.index, bars.values, color=[ORANGE if k == "w_b" else CEMENT_GREY for k in bars.index])
axes[1].set(title="Variance of ln(28-day strength) explained by one variable", xlabel="R²")
fig.tight_layout()
savefig(fig, "4_1_abrams")
plt.show()

learn("concrete", f"At 28 days, the water-to-binder ratio alone explains R² = {r2_single['w_b']:.2f} of log-strength. The figures are "
      f"{r2_single['w_c']:.2f} for water/cement, {r2_single['cement']:.2f} for cement content and {r2_single['water']:.2f} for water content. "
      + ("Abrams' century-old rule holds in this data: a ratio carries more information than any single ingredient."
         if r2_single["w_b"] >= max(r2_single["cement"], r2_single["water"]) else
         "In this data a single ingredient is at least as informative as the ratio, so Abrams' rule does not fully hold."))

### 4.2 Strength gain with age
Cement hydration continues for months, so strength rises quickly at first and then levels off. For mixes tested at 28 days and at other ages, each strength is divided by the same mix's 28-day strength. This isolates the effect of age from differences between mixes. The comparison splits pure-cement mixes from those containing supplementary cementitious materials (SCMs: slag or fly ash), which react more slowly.

In [ ]:
s28 = d28.groupby("mix_id")["strength"].mean()
rel = concrete[concrete["mix_id"].isin(s28.index)].copy()
rel["relative"] = rel["strength"] / rel["mix_id"].map(s28)
rel["binder_type"] = np.where(rel["scm_frac"] > 0.2, "with SCM (>20%)", "mostly cement")
gain = rel.groupby(["binder_type", "age"])["relative"].agg(["median", "count"]).reset_index()
gain = gain[gain["count"] >= 5]

fig, ax = plt.subplots(figsize=(10, 4.8))
for (bt, g), c in zip(gain.groupby("binder_type"), [CEMENT_GREY, GREEN]):
    ax.plot(g["age"], g["median"], "o-", color=c, lw=2, label=bt)
ax.axhline(1, ls=":", c=INK)
ax.set(xscale="log", title="Strength relative to the same mix at 28 days", xlabel="age [days, log scale]", ylabel="f(t) / f(28)")
ax.legend()
savefig(fig, "4_2_strength_gain")
plt.show()


def ratio_at(age, bt):
    s = rel[(rel["age"] == age) & (rel["binder_type"] == bt)]["relative"]
    return s.median() if len(s) >= 3 else np.nan


r7 = {bt: ratio_at(7, bt) for bt in rel["binder_type"].unique()}
late_age = max([a for a in [90, 56, 100, 180] if rel[rel["age"] == a]["binder_type"].nunique() == 2] or [np.nan])
rL = {bt: ratio_at(late_age, bt) for bt in rel["binder_type"].unique()} if np.isfinite(late_age) else {}
text = f"At 7 days, mostly-cement mixes reach {r7.get('mostly cement', np.nan):.0%} of their 28-day strength and SCM mixes {r7.get('with SCM (>20%)', np.nan):.0%}."
if rL:
    text += f" At {int(late_age)} days they reach {rL.get('mostly cement', np.nan):.0%} and {rL.get('with SCM (>20%)', np.nan):.0%} respectively."
early_gap = r7.get("mostly cement", np.nan) - r7.get("with SCM (>20%)", np.nan)
late_gap = rL.get("with SCM (>20%)", np.nan) - rL.get("mostly cement", np.nan) if rL else np.nan
if early_gap > 0.03 and (not np.isfinite(late_gap) or late_gap > 0.03):
    text += " Slag and fly ash give lower early strength and gain more later, so the 28-day test undervalues SCM mixes."
elif np.isfinite(early_gap):
    text += " The difference in the rate of strength gain between the two binder types is small in this data."
learn("concrete", text)

### 4.3 Correlations: ingredients or ratios?
Spearman correlations of the raw ingredients and of the engineering ratios with strength, calculated separately for each age group. The original analysis used a single Pearson matrix over all ages. That mixes the effect of age with the effect of the mix design.

In [ ]:
cols = INGREDIENTS + ["w_b", "w_c", "scm_frac"]
corr_age = pd.DataFrame({g: concrete[concrete["age_group"] == g][cols + ["strength"]].corr(method="spearman")["strength"].drop("strength")
                         for g in AGE_COLORS})
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(corr_age, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax, cbar=False, linewidths=0.5)
ax.set_title("Spearman correlation with strength, by age group")
savefig(fig, "4_3_correlations")
plt.show()
top = corr_age["standard (14–28 d)"].abs().sort_values(ascending=False)
learn("concrete", f"At standard ages, the variables most strongly associated with strength are {', '.join(top.index[:3])} "
      f"(|ρ| = {', '.join(f'{v:.2f}' for v in top.values[:3])}). Because correlations differ between age groups, age has to be "
      "modelled together with the mix design and not as a separate additive effect.")

## 5 · Feature Engineering
Two feature sets are compared:

| Set | Features |
|---|---|
| **raw** | the 7 ingredients and age, as in the original analysis |
| **engineered** | raw features plus binder content, w/b, w/c, SCM fraction, superplasticizer per binder, aggregate-to-binder ratio, fine-aggregate fraction and ln(age) |

The features are computed inside each model pipeline from the raw inputs. This lets the mix optimiser and the interpretation tools change raw ingredient amounts while the ratios stay consistent with them.

In [ ]:
def engineer(df, feature_set="engineered"):
    out = df[RAW_INPUTS].copy()
    if feature_set == "engineered":
        binder = out["cement"] + out["slag"] + out["fly_ash"]
        agg = out["coarse_agg"] + out["fine_agg"]
        out["binder"] = binder
        out["w_b"] = out["water"] / binder
        out["w_c"] = out["water"] / out["cement"]
        out["scm_frac"] = (out["slag"] + out["fly_ash"]) / binder
        out["sp_per_binder"] = out["superplasticizer"] / binder
        out["agg_binder"] = agg / binder
        out["fine_frac"] = out["fine_agg"] / agg
        out["log_age"] = np.log(out["age"])
    return out


FEATURE_SETS = ["raw", "engineered"]
for fs in FEATURE_SETS:
    print(f"{fs:<10} ({engineer(concrete.head(), fs).shape[1]}): {', '.join(engineer(concrete.head(), fs).columns)}")

## 6 · Validation Design
- **Hold-out set:** 20 % of the **mixes**, with all their test ages, are set aside for the final evaluation in §9.
- **Cross-validation:** grouped 5-fold CV repeated 3 times, where each repeat shuffles which mixes fall into which fold.

The first comparison repeats the original random row split. A default random forest is scored under random K-fold and under mix-grouped K-fold, and the gap between the two shows how much of its apparent accuracy comes from having seen the same mix at other ages.

In [ ]:
def grouped_splits(groups, n_splits, repeats, seed):
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    rng = np.random.default_rng(seed)
    splits = []
    for _ in range(repeats):
        fold_of = dict(zip(rng.permutation(uniq), np.arange(len(uniq)) % n_splits))
        f = np.array([fold_of[g] for g in groups])
        for k in range(n_splits):
            splits.append((np.where(f != k)[0], np.where(f == k)[0]))
    return splits


gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG["test_fraction"], random_state=SEED)
tr_idx, te_idx = next(gss.split(concrete, groups=concrete["mix_id"]))
train, test = concrete.iloc[tr_idx].reset_index(drop=True), concrete.iloc[te_idx].reset_index(drop=True)
X_train, y_train, X_test, y_test = train[RAW_INPUTS], train["strength"], test[RAW_INPUTS], test["strength"]
CV = grouped_splits(train["mix_id"], CONFIG["cv_splits"], CONFIG["cv_repeats"], SEED)
assert not set(train["mix_id"]) & set(test["mix_id"])
display(pd.DataFrame({"tests": [len(train), len(test)], "mixes": [train["mix_id"].nunique(), test["mix_id"].nunique()],
                      "mean strength": [y_train.mean(), y_test.mean()]}, index=["train", "test"]).style.format(precision=1))

probe = Pipeline([("f", FunctionTransformer(engineer, kw_args={"feature_set": "raw"})),
                  ("m", RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1))])
random_cv = cross_validate(probe, X_train, y_train, cv=KFold(5, shuffle=True, random_state=SEED), scoring=("neg_root_mean_squared_error", "r2"))
group_cv = cross_validate(probe, X_train, y_train, cv=grouped_splits(train["mix_id"], 5, 1, SEED), scoring=("neg_root_mean_squared_error", "r2"))
leak = pd.DataFrame({"random K-fold": [-random_cv["test_neg_root_mean_squared_error"].mean(), random_cv["test_r2"].mean()],
                     "mix-grouped K-fold": [-group_cv["test_neg_root_mean_squared_error"].mean(), group_cv["test_r2"].mean()]},
                    index=["RMSE [MPa]", "R²"])
display(leak.style.format(precision=3))
learn("process", f"Random row splitting gives RMSE {leak.loc['RMSE [MPa]', 'random K-fold']:.2f} MPa. Keeping each mix on one side gives "
      f"{leak.loc['RMSE [MPa]', 'mix-grouped K-fold']:.2f} MPa. "
      + ("The random split overstates accuracy for new mixes because the model has already seen the test mixes at other ages."
         if leak.loc['RMSE [MPa]', 'mix-grouped K-fold'] > leak.loc['RMSE [MPa]', 'random K-fold'] * 1.05
         else "The difference is small here, but mix-grouped validation remains the correct protocol."))

## 7 · Model Benchmark

### 7.1 Physical benchmark: Abrams' law with strength gain
The physics-based model combines Abrams' law with the hyperbolic strength-gain function of ACI 209 and a correction for SCM content:

$$f_c(t) = A\,e^{-k\,(w/b)}\cdot\frac{t}{a + b\,t}\cdot\left(1 + c\cdot \text{SCM}\right)$$

Its five parameters are fitted by nonlinear least squares. It uses only three quantities, w/b, age and SCM fraction, so it is the most interpretable model in the benchmark.

In [ ]:
class AbramsAgeModel(BaseEstimator, RegressorMixin):
    def _terms(self, X):
        binder = X["cement"] + X["slag"] + X["fly_ash"]
        return (X["water"] / binder).to_numpy(), X["age"].to_numpy(float), ((X["slag"] + X["fly_ash"]) / binder).to_numpy()

    @staticmethod
    def _f(Z, A, k, a, b, c):
        wb, t, scm = Z
        return A * np.exp(-k * wb) * t / (a + b * t) * (1 + c * scm)

    def fit(self, X, y):
        Z = self._terms(X)
        self.params_, _ = optimize.curve_fit(self._f, Z, np.asarray(y, float), p0=[150, 2.5, 4, 0.9, 0.0],
                                             bounds=([1, 0, 0.01, 0.1, -1], [2000, 10, 100, 5, 2]), maxfev=20000)
        return self

    def predict(self, X):
        return self._f(self._terms(X), *self.params_)


phys = AbramsAgeModel().fit(X_train, y_train)
print("Fitted parameters (A, k, a, b, c):", np.round(phys.params_, 3))

### 7.2 Model library

| Key | Model | Family |
|---|---|---|
| PHYS | Abrams + ACI strength gain | physics |
| LIN / LASSO / RIDGE | linear regressions (original) | linear |
| DT | decision tree (original) | tree |
| RF | random forest (original) | tree ensemble |
| GBR / XGB / LGBM / CAT | gradient boosting | boosting |
| SVR / GP | support vector regression, Gaussian process | kernel |
| KNN | k-nearest neighbours | instance-based |
| MLP | neural network | neural |

In [ ]:
MODEL_NAMES = {"PHYS": "Abrams + age (physics)", "LIN": "Linear regression", "LASSO": "Lasso", "RIDGE": "Ridge",
               "DT": "Decision tree", "RF": "Random forest", "GBR": "Gradient boosting", "XGB": "XGBoost", "LGBM": "LightGBM",
               "CAT": "CatBoost", "SVR": "SVR", "GP": "Gaussian process", "KNN": "k-NN", "MLP": "MLP"}
FAMILY = {"PHYS": "physics", "LIN": "linear", "LASSO": "linear", "RIDGE": "linear", "DT": "tree", "RF": "tree ensemble",
          "GBR": "boosting", "XGB": "boosting", "LGBM": "boosting", "CAT": "boosting", "SVR": "kernel", "GP": "kernel",
          "KNN": "instance-based", "MLP": "neural"}
FAMILY_COLORS = {"physics": ORANGE, "linear": CEMENT_GREY, "tree": "#A3A3A3", "tree ensemble": GREEN, "boosting": BLUE,
                 "kernel": "#7C3AED", "instance-based": "#D97706", "neural": "#BE123C"}


def base_model(key, n_features):
    scaled_target = lambda est: TransformedTargetRegressor(regressor=est, transformer=StandardScaler())
    return {
        "LIN": lambda: LinearRegression(),
        "LASSO": lambda: Lasso(alpha=1.0, max_iter=50000),
        "RIDGE": lambda: Ridge(alpha=1.0),
        "DT": lambda: DecisionTreeRegressor(random_state=SEED),
        "RF": lambda: RandomForestRegressor(n_estimators=400, n_jobs=1, random_state=SEED),
        "GBR": lambda: GradientBoostingRegressor(random_state=SEED),
        "XGB": lambda: XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=5, subsample=0.9, colsample_bytree=0.9,
                                    n_jobs=1, random_state=SEED),
        "LGBM": lambda: LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31, min_child_samples=5, n_jobs=1,
                                      random_state=SEED, verbose=-1),
        "CAT": lambda: CatBoostRegressor(iterations=800, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0,
                                         thread_count=1, allow_writing_files=False),
        "SVR": lambda: scaled_target(SVR(C=10, epsilon=0.05, gamma="scale")),
        "GP": lambda: GaussianProcessRegressor(ConstantKernel(1.0) * RBF(np.ones(n_features), (1e-2, 1e3)) + WhiteKernel(0.1, (1e-5, 1)),
                                               normalize_y=True, n_restarts_optimizer=1, random_state=SEED),
        "KNN": lambda: KNeighborsRegressor(n_neighbors=5, weights="distance"),
        "MLP": lambda: scaled_target(MLPRegressor(hidden_layer_sizes=(64, 32), alpha=1e-3, max_iter=4000, random_state=SEED)),
    }[key]()


def build(key, feature_set="engineered"):
    if key == "PHYS":
        return AbramsAgeModel()
    n = engineer(concrete.head(), feature_set).shape[1]
    return Pipeline([("features", FunctionTransformer(engineer, kw_args={"feature_set": feature_set})),
                     ("scale", StandardScaler()), ("model", base_model(key, n))])


def prefix(key):
    return "model__regressor__" if key in ("SVR", "MLP") else "model__"

### 7.3 Benchmark under mix-grouped cross-validation
All models are run on both feature sets. The physics model's result does not depend on the feature set.

In [ ]:
SCORING = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}
bench, folds = [], {}
t0 = time.time()
for fs in FEATURE_SETS:
    for key in MODEL_NAMES:
        if key == "PHYS" and fs == "raw":
            continue
        res = cross_validate(build(key, fs), X_train, y_train, cv=CV, scoring=SCORING, n_jobs=-1)
        folds[(fs, key)] = -res["test_rmse"]
        bench.append({"feature_set": fs, "key": key, "model": MODEL_NAMES[key], "family": FAMILY[key],
                      "RMSE": -res["test_rmse"].mean(), "RMSE sd": res["test_rmse"].std(), "MAE": -res["test_mae"].mean(),
                      "R²": res["test_r2"].mean()})
bench = pd.DataFrame(bench)
print(f"Benchmark finished in {time.time() - t0:.0f} s")
piv = bench.pivot(index="model", columns="feature_set", values="RMSE")
piv["raw"] = piv["raw"].fillna(piv["engineered"])
piv["gain from engineering"] = piv["raw"] - piv["engineered"]
display(piv.sort_values("engineered").style.format(precision=3).background_gradient(subset=["engineered"], cmap="YlGn_r"))
bench.to_csv(DIRS["tables"] / "7_3_benchmark.csv", index=False)

In [ ]:
eng = bench[bench["feature_set"] == "engineered"].sort_values("RMSE")
fig, ax = plt.subplots(figsize=(11, 6))
data_ = [folds[("engineered", k)] for k in eng["key"]]
bp = ax.boxplot(data_, vert=False, patch_artist=True, widths=0.6, medianprops={"color": INK})
for patch, k in zip(bp["boxes"], eng["key"]):
    patch.set_facecolor(FAMILY_COLORS[FAMILY[k]])
    patch.set_alpha(0.8)
ax.set_yticks(range(1, len(eng) + 1), eng["model"])
ax.invert_yaxis()
ax.set(title="Mix-grouped CV RMSE (engineered features)", xlabel="RMSE [MPa]")
savefig(fig, "7_3_benchmark")
plt.show()

best_key = eng["key"].iloc[0]
linear_gain = piv.loc[["Linear regression", "Ridge", "Lasso"], "gain from engineering"].mean()
tree_gain = piv.loc[["Random forest", "XGBoost", "LightGBM", "CatBoost"], "gain from engineering"].mean()
learn("modelling", f"Under mix-grouped CV the best default model is {MODEL_NAMES[best_key]} (RMSE {eng['RMSE'].iloc[0]:.2f} MPa). The physics model reaches "
      f"{eng.set_index('key').loc['PHYS', 'RMSE']:.2f} MPa with five parameters, against {eng.set_index('key').loc['LIN', 'RMSE']:.2f} for linear regression on raw ingredients. "
      f"Engineered ratios reduce RMSE by {linear_gain:.2f} MPa on average for linear models and by {tree_gain:.2f} MPa for tree ensembles. "
      + ("Ratios help the linear models far more than the tree ensembles, which can partly learn such interactions themselves."
         if linear_gain > tree_gain + 0.2 else "The ratios help both model families by a similar amount."))

## 8 · Hyperparameter Tuning
Each ML model is tuned on the engineered feature set by randomised search over the same mix-grouped CV splits, minimising RMSE. The physics model and the Gaussian process fit their parameters directly, the GP by maximising the marginal likelihood.

In [ ]:
SPACES = {
    "LASSO": {"alpha": loguniform(1e-3, 5)},
    "RIDGE": {"alpha": loguniform(1e-3, 1e3)},
    "DT": {"max_depth": [None, 6, 8, 10, 14, 20], "min_samples_leaf": randint(1, 10)},
    "RF": {"n_estimators": randint(200, 800), "max_depth": [None, 10, 16, 24], "min_samples_leaf": randint(1, 5),
           "max_features": [0.33, 0.5, 0.8, 1.0]},
    "GBR": {"n_estimators": randint(200, 1200), "learning_rate": loguniform(0.01, 0.2), "max_depth": randint(2, 7),
            "subsample": uniform(0.6, 0.4), "min_samples_leaf": randint(1, 10)},
    "XGB": {"n_estimators": randint(300, 1500), "learning_rate": loguniform(0.01, 0.2), "max_depth": randint(3, 9),
            "min_child_weight": loguniform(0.5, 10), "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.5, 0.5),
            "reg_lambda": loguniform(1e-2, 10)},
    "LGBM": {"n_estimators": randint(300, 1500), "learning_rate": loguniform(0.01, 0.2), "num_leaves": randint(8, 64),
             "min_child_samples": randint(3, 30), "subsample": uniform(0.6, 0.4), "subsample_freq": [1],
             "colsample_bytree": uniform(0.5, 0.5), "reg_lambda": loguniform(1e-3, 10)},
    "CAT": {"iterations": randint(400, 1500), "learning_rate": loguniform(0.02, 0.2), "depth": randint(4, 9),
            "l2_leaf_reg": loguniform(1, 10)},
    "SVR": {"C": loguniform(0.5, 500), "epsilon": loguniform(1e-3, 0.3), "gamma": loguniform(1e-3, 1)},
    "KNN": {"n_neighbors": randint(2, 20), "weights": ["uniform", "distance"], "p": [1, 2]},
    "MLP": {"hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64), (128, 64, 32)], "alpha": loguniform(1e-5, 1),
            "learning_rate_init": loguniform(1e-4, 1e-2)},
}
tuned, rows = {}, []
for key in MODEL_NAMES:
    t0 = time.time()
    base_rmse = eng.set_index("key").loc[key, "RMSE"]
    if key in SPACES:
        n_iter = max(3, CONFIG["search_iter"] // (2 if key in ("CAT", "MLP", "GBR") else 1))
        search = RandomizedSearchCV(build(key), {prefix(key) + k: v for k, v in SPACES[key].items()}, n_iter=n_iter, cv=CV,
                                    scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=SEED)
        search.fit(X_train, y_train)
        tuned[key], cv_rmse = search.best_estimator_, -search.best_score_
        params = {k.split("__")[-1]: (round(v, 4) if isinstance(v, float) else v) for k, v in search.best_params_.items()}
    else:
        tuned[key], cv_rmse, params = build(key).fit(X_train, y_train), base_rmse, {}
    rows.append({"model": MODEL_NAMES[key], "key": key, "default CV RMSE": base_rmse, "tuned CV RMSE": cv_rmse,
                 "gain": base_rmse - cv_rmse, "seconds": time.time() - t0, "parameters": params})
    print(f"{MODEL_NAMES[key]:<26} {cv_rmse:.3f} MPa")
tuning = pd.DataFrame(rows).set_index("model").sort_values("tuned CV RMSE")
display(tuning.drop(columns="key").style.format(precision=3, subset=["default CV RMSE", "tuned CV RMSE", "gain"]).format("{:.0f}", subset=["seconds"]))
tuning.to_csv(DIRS["tables"] / "8_tuning.csv")
CHAMPION = tuning["key"].iloc[0]
learn("modelling", f"Tuning lowers CV RMSE by {tuning['gain'].median():.2f} MPa at the median, and by at most {tuning['gain'].max():.2f} MPa "
      f"({tuning['gain'].idxmax()}). The champion is {MODEL_NAMES[CHAMPION]} with RMSE {tuning['tuned CV RMSE'].iloc[0]:.2f} MPa.")

## 9 · Hold-out Evaluation

### 9.1 Scores on unseen mixes
Every model is scored on the held-out mixes. Confidence intervals come from a **cluster bootstrap** that resamples whole mixes rather than individual tests.

In [ ]:
def cluster_boot(y, p, g, n=CONFIG["bootstrap"]):
    frame = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(p), "g": np.asarray(g)})
    parts = [v for _, v in frame.groupby("g")]
    r = np.random.default_rng(SEED)
    out = []
    for _ in range(n):
        s = pd.concat([parts[i] for i in r.integers(0, len(parts), len(parts))])
        out.append(rmse(s["y"], s["p"]))
    return np.percentile(out, [2.5, 97.5])


pred = {k: m.predict(X_test) for k, m in tuned.items()}
hold = []
for k, p in pred.items():
    lo, hi = cluster_boot(y_test, p, test["mix_id"])
    hold.append({"model": MODEL_NAMES[k], "key": k, "family": FAMILY[k], "RMSE": rmse(y_test, p), "RMSE 95% CI": f"{lo:.2f}–{hi:.2f}",
                 "MAE": mean_absolute_error(y_test, p), "R²": r2_score(y_test, p),
                 "MAPE %": 100 * np.mean(np.abs((y_test - p) / y_test))})
holdout = pd.DataFrame(hold).set_index("model").sort_values("RMSE")
display(holdout.drop(columns="key").style.format(precision=3).background_gradient(subset=["RMSE"], cmap="YlGn_r"))
holdout.to_csv(DIRS["tables"] / "9_holdout.csv")
ch = holdout[holdout["key"] == CHAMPION].iloc[0]
learn("modelling", f"On unseen mixes, {MODEL_NAMES[CHAMPION]} reaches RMSE {ch['RMSE']:.2f} MPa (95% CI {ch['RMSE 95% CI']}) and R² {ch['R²']:.3f}. "
      f"The original random-split random forest reported RMSE 5.08 MPa. That figure is not directly comparable, because its test mixes also appeared in training at other ages.")

### 9.2 Error structure
Measured vs predicted strength for the champion and the physics model. The champion's errors are also broken down by age group and by strength class, to show where the model is least reliable.

In [ ]:
p = pred[CHAMPION]
fig, axes = plt.subplots(1, 3, figsize=(20, 5.2))
for ax, k in zip(axes[:2], [CHAMPION, "PHYS"]):
    for g, c in AGE_COLORS.items():
        m = (test["age_group"] == g).to_numpy()
        ax.scatter(y_test[m], pred[k][m], s=22, color=c, alpha=0.8, label=g, edgecolor="k", lw=0.2)
    lim = [0, max(y_test.max(), pred[k].max()) * 1.05]
    ax.plot(lim, lim, "--", c=INK, lw=1)
    ax.set(title=f"{MODEL_NAMES[k]} (RMSE {rmse(y_test, pred[k]):.2f})", xlabel="measured [MPa]", ylabel="predicted [MPa]", xlim=lim, ylim=lim)
axes[0].legend(fontsize=8)
err = test.assign(abs_err=np.abs(y_test - p), strength_class=pd.cut(y_test, [0, 20, 40, 60, 100], labels=["<20", "20–40", "40–60", ">60"]))
seg = err.groupby(["age_group", "strength_class"], observed=True)["abs_err"].mean().unstack()
sns.heatmap(seg, annot=True, fmt=".1f", cmap="Oranges", ax=axes[2], cbar_kws={"label": "MAE [MPa]"})
axes[2].set(title=f"{MODEL_NAMES[CHAMPION]}: MAE by age and strength class", xlabel="measured strength class [MPa]", ylabel="")
fig.tight_layout()
savefig(fig, "9_2_errors")
plt.show()
by_age = err.groupby("age_group", observed=True)["abs_err"].mean()
learn("modelling", f"Champion MAE by age group: " + ", ".join(f"{g} {v:.2f} MPa" for g, v in by_age.items()) + ". "
      f"The largest errors occur at {by_age.idxmax()} ages.")

### 9.3 Prediction intervals
A strength prediction is only useful for design if its uncertainty is known. Two approaches are compared:
- **Split-conformal intervals** for the champion, with the half-width taken from its out-of-fold residuals under mix-grouped CV;
- the Gaussian process's own predictive distribution.

Both are checked for coverage on the unseen mixes, overall and by age group.

In [ ]:
alpha = CONFIG["conformal_alpha"]
oof = cross_val_predict(clone(tuned[CHAMPION]), X_train, y_train, cv=grouped_splits(train["mix_id"], 5, 1, SEED), n_jobs=-1)
res_sorted = np.sort(np.abs(y_train - oof))
Q_HALF = float(res_sorted[min(len(res_sorted) - 1, int(np.ceil((len(res_sorted) + 1) * (1 - alpha))) - 1)])
z = stats.norm.ppf(1 - alpha / 2)
gp_mean, gp_sd = tuned["GP"].predict(X_test, return_std=True)
cov = pd.DataFrame({
    "conformal covered": np.abs(y_test - pred[CHAMPION]) <= Q_HALF,
    "GP covered": np.abs(y_test - gp_mean) <= z * gp_sd,
    "GP half-width": z * gp_sd, "age_group": test["age_group"]})
cov_tab = cov.groupby("age_group", observed=True).agg(conformal=("conformal covered", "mean"), gp=("GP covered", "mean"),
                                                    gp_width=("GP half-width", "mean"))
cov_tab.loc["all"] = [cov["conformal covered"].mean(), cov["GP covered"].mean(), cov["GP half-width"].mean()]
display(cov_tab.rename(columns={"conformal": f"conformal coverage (±{Q_HALF:.1f} MPa)", "gp": "GP coverage", "gp_width": "GP mean half-width"})
        .style.format("{:.1%}", subset=[f"conformal coverage (±{Q_HALF:.1f} MPa)", "GP coverage"]).format("{:.2f}", subset=["GP mean half-width"]))
learn("modelling", f"A {1 - alpha:.0%} conformal interval for the champion is ±{Q_HALF:.1f} MPa and covers {cov['conformal covered'].mean():.0%} of unseen tests. "
      f"The Gaussian process gives intervals whose width varies from test to test, with {cov['GP covered'].mean():.0%} coverage. "
      "A design that relies on a predicted strength should subtract a margin of this size.")

## 10 · Interpretation

### 10.1 Ingredient importance (composition-aware permutation)
Each raw input is shuffled in turn across the test set. The ratios derived from it (w/b, SCM fraction and so on) are recomputed from the shuffled values, so the measured increase in RMSE is the total effect of that ingredient.

In [ ]:
def raw_permutation(model, X, y, repeats=20):
    r = np.random.default_rng(SEED)
    base = rmse(y, model.predict(X))
    out = {}
    for c in RAW_INPUTS:
        vals = []
        for _ in range(repeats):
            Xp = X.copy()
            Xp[c] = r.permutation(Xp[c].to_numpy())
            vals.append(rmse(y, model.predict(Xp)) - base)
        out[c] = np.mean(vals)
    return pd.Series(out)


imp = pd.DataFrame({MODEL_NAMES[k]: raw_permutation(tuned[k], X_test, y_test) for k in dict.fromkeys([CHAMPION, "RF", "PHYS"])})
imp = imp.loc[imp.mean(axis=1).sort_values().index]
fig, ax = plt.subplots(figsize=(10, 5))
imp.plot(kind="barh", ax=ax, width=0.8, color=[BLUE, GREEN, ORANGE][:imp.shape[1]])
ax.set(title="Increase in test RMSE when an input is shuffled", xlabel="ΔRMSE [MPa]")
savefig(fig, "10_1_importance")
plt.show()
top3 = imp[MODEL_NAMES[CHAMPION]].sort_values(ascending=False)
learn("concrete", f"For the champion, the most influential inputs are {', '.join(top3.index[:3])}, and the least influential are {', '.join(top3.index[-2:])}. "
      "The original conclusion that fly ash and the aggregates matter least can be checked against this list. Impurity importances, which it relied on, are biased towards variables with many distinct values.")

### 10.2 Effect of water and cement at fixed age
The individual conditional expectation curves show how predicted 28-day strength responds when water or cement is changed while every other ingredient is held at its value in a given test mix. Because the ratios are recomputed, these curves show how the model has learned Abrams' law.

In [ ]:
base28 = X_test.drop_duplicates(subset=INGREDIENTS).assign(age=28).sample(min(60, X_test[INGREDIENTS].drop_duplicates().shape[0]), random_state=SEED)
fig, axes = plt.subplots(1, 3, figsize=(20, 4.5), sharey=True)
effects = {}
for ax, col in zip(axes, ["water", "cement", "superplasticizer"]):
    grid = np.linspace(*concrete[col].quantile([0.02, 0.98]), 30)
    curves = np.column_stack([tuned[CHAMPION].predict(base28.assign(**{col: v})) for v in grid])
    ax.plot(grid, curves.T, color=BLUE, alpha=0.12)
    ax.plot(grid, curves.mean(axis=0), color=INK, lw=2.5)
    ax.set(title=f"28-day strength vs {col}", xlabel=f"{col} [kg/m³]", ylabel="predicted strength [MPa]")
    step = {"water": 10, "cement": 10, "superplasticizer": 1}[col]
    effects[col] = np.mean(tuned[CHAMPION].predict(base28.assign(**{col: base28[col] + step})) - tuned[CHAMPION].predict(base28))
fig.tight_layout()
savefig(fig, "10_2_ice")
plt.show()
learn("concrete", f"At 28 days, the champion predicts {effects['water']:+.2f} MPa per extra 10 kg of water, {effects['cement']:+.2f} MPa per extra 10 kg of cement "
      f"and {effects['superplasticizer']:+.2f} MPa per extra 1 kg of superplasticizer, averaged over test mixes. "
      + ("These signs agree with concrete technology: more water weakens concrete, while cement and superplasticizer strengthen it (superplasticizer by allowing less water)."
         if effects["water"] < 0 < effects["cement"] else "Some signs contradict standard concrete technology and deserve scrutiny."))

## 11 · Strength-Gain Curves
`strength_curve()` predicts strength from 1 to 365 days for any mix, with the conformal band from §9.3. It is shown for four test mixes that were measured at several ages, together with their measured strengths and the physics model's curve.

In [ ]:
AGES = np.unique(np.r_[np.geomspace(1, 365, 40), [3, 7, 14, 28, 56, 90, 180, 365]])


def strength_curve(mix, model=None):
    model = model or tuned[CHAMPION]
    frame = pd.DataFrame([{**{c: mix[c] for c in INGREDIENTS}, "age": a} for a in AGES])
    pred_ = model.predict(frame[RAW_INPUTS])
    return pd.DataFrame({"age": AGES, "strength": pred_, "lower": pred_ - Q_HALF, "upper": pred_ + Q_HALF})


multi = test.groupby("mix_id").filter(lambda g: g["age"].nunique() >= 3)
chosen = multi.groupby("mix_id")["scm_frac"].first().sort_values()
chosen = list(dict.fromkeys([chosen.index[0], chosen.index[-1], *chosen.index[len(chosen) // 3: len(chosen) // 3 + 2]]))[:4]
fig, axes = plt.subplots(1, max(1, len(chosen)), figsize=(5.2 * max(1, len(chosen)), 4.3), sharey=True, squeeze=False)
for ax, mid in zip(axes[0], chosen):
    g = test[test["mix_id"] == mid].sort_values("age")
    mix = g.iloc[0]
    cur = strength_curve(mix)
    ax.fill_between(cur["age"], cur["lower"], cur["upper"], color=BLUE, alpha=0.12)
    ax.plot(cur["age"], cur["strength"], color=BLUE, lw=2, label=MODEL_NAMES[CHAMPION])
    ax.plot(cur["age"], strength_curve(mix, tuned["PHYS"])["strength"], "--", color=ORANGE, lw=1.8, label="physics")
    ax.scatter(g["age"], g["strength"], color=INK, zorder=5, label="measured")
    ax.set(xscale="log", title=f"w/b {mix['w_b']:.2f} · SCM {mix['scm_frac']:.0%}", xlabel="age [days]")
axes[0, 0].set_ylabel("strength [MPa]")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
savefig(fig, "11_strength_curves")
plt.show()

## 12 · Mix Design Optimiser
Given a target 28-day strength, the optimiser searches for the mix that minimises **cost** or **CO₂**. It uses differential evolution, a global optimiser that does not need gradients and therefore works with tree ensembles. Every candidate mix has to satisfy all of the following:

1. **Guaranteed strength:** the lower conformal bound of the predicted strength must reach the target, so there is a safety margin;
2. **Yield:** the absolute volume of the ingredients plus air must be 1 m³ ± 2 %;
3. **Ingredient bounds:** each ingredient must lie within the range seen in the training data;
4. **Applicability domain:** the mix must be no further from its nearest training mix than 95 % of training mixes are from theirs, and its w/b and SCM fraction must lie within the range of the training mixes. This stops the optimiser from exploiting the model in regions with no data.

The recommended mixes are **model-based suggestions**. They must be verified by trial batches before use.

In [ ]:
price = np.array([CONFIG["price_per_kg"][c] for c in INGREDIENTS])
co2 = np.array([CONFIG["co2_per_kg"][c] for c in INGREDIENTS])
dens_v = np.array([CONFIG["density_kg_m3"][c] for c in INGREDIENTS])
train_mixes = train.drop_duplicates("mix_id")[INGREDIENTS]
BOUNDS = [(train_mixes[c].min(), train_mixes[c].max()) for c in INGREDIENTS]
dom_scaler = StandardScaler().fit(train_mixes)
dom_nn = NearestNeighbors(n_neighbors=2).fit(dom_scaler.transform(train_mixes))
DOMAIN_LIMIT = np.quantile(dom_nn.kneighbors(dom_scaler.transform(train_mixes))[0][:, 1], 0.95)
tm_binder = train_mixes["cement"] + train_mixes["slag"] + train_mixes["fly_ash"]
WB_RANGE = ((train_mixes["water"] / tm_binder).quantile(0.01), (train_mixes["water"] / tm_binder).quantile(0.99))
SCM_MAX = ((train_mixes["slag"] + train_mixes["fly_ash"]) / tm_binder).quantile(0.99)


def evaluate_mixes(M):
    frame = pd.DataFrame(M, columns=INGREDIENTS).assign(age=28)
    strength_ = tuned[CHAMPION].predict(frame[RAW_INPUTS])
    volume = (M / dens_v).sum(axis=1) + CONFIG["air_content"]
    dist = dom_nn.kneighbors(dom_scaler.transform(pd.DataFrame(M, columns=INGREDIENTS)), n_neighbors=1)[0][:, 0]
    return strength_, volume, dist


def design(target, objective="cost"):
    weights = price if objective == "cost" else co2

    def f(x):
        M = np.atleast_2d(x.T) if x.ndim == 2 else x[None, :]
        s, v, d = evaluate_mixes(M)
        binder = M[:, 0] + M[:, 1] + M[:, 2]
        wb, scm = M[:, 3] / binder, (M[:, 1] + M[:, 2]) / binder
        penalty = (np.maximum(0, target - (s - Q_HALF)) * 50 + np.maximum(0, np.abs(v - 1) - 0.02) * 5000
                   + np.maximum(0, d - DOMAIN_LIMIT) * 200 + (np.maximum(0, WB_RANGE[0] - wb) + np.maximum(0, wb - WB_RANGE[1])) * 2000
                   + np.maximum(0, scm - SCM_MAX) * 2000)
        val = M @ weights + penalty
        return val if x.ndim == 2 else val[0]

    res = optimize.differential_evolution(f, BOUNDS, seed=SEED, maxiter=CONFIG["de_maxiter"], popsize=20, tol=1e-6,
                                          polish=False, vectorized=True, updating="deferred")
    m = res.x
    s, v, d = evaluate_mixes(m[None, :])
    binder = m[0] + m[1] + m[2]
    return {"target MPa": target, "objective": objective, **{c: round(val, 1) for c, val in zip(INGREDIENTS, m)},
            "w/b": m[3] / binder, "SCM %": 100 * (m[1] + m[2]) / binder, "predicted MPa": s[0], "guaranteed MPa": s[0] - Q_HALF,
            "volume m³": v[0], "cost / m³": m @ price, "CO₂ kg / m³": m @ co2, "domain distance": d[0] / DOMAIN_LIMIT,
            "feasible": (s[0] - Q_HALF >= target - 0.5) and abs(v[0] - 1) <= 0.025 and d[0] <= DOMAIN_LIMIT * 1.02
                        and WB_RANGE[0] - 0.01 <= m[3] / binder <= WB_RANGE[1] + 0.01 and (m[1] + m[2]) / binder <= SCM_MAX + 0.01}


t0 = time.time()
designs = pd.DataFrame([design(t, obj) for obj in ["cost", "co2"] for t in CONFIG["design_targets_mpa"]])
print(f"Optimisation finished in {time.time() - t0:.0f} s")
display(designs.style.format(precision=2).format(precision=0, subset=INGREDIENTS)
        .apply(lambda r: ["background-color: #FEE2E2" if not r["feasible"] else "" for _ in r], axis=1))
designs.to_csv(DIRS["tables"] / "12_mix_designs.csv", index=False)

In [ ]:
typical = train[train["age"] == 28].copy()
typical["cost"] = typical[INGREDIENTS].to_numpy() @ price
typical["co2"] = typical[INGREDIENTS].to_numpy() @ co2
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (col, obj, lab) in zip(axes, [("cost", "cost", "cost per m³"), ("co2", "co2", "CO₂ [kg per m³]")]):
    ax.scatter(typical["strength"], typical[col], s=16, color=SAND, edgecolor=CEMENT_GREY, lw=0.4, label="training mixes (28 d, measured)")
    d = designs[(designs["objective"] == obj) & designs["feasible"]]
    ax.plot(d["guaranteed MPa"], d["cost / m³" if col == "cost" else "CO₂ kg / m³"], "o-", color=ORANGE if obj == "cost" else GREEN,
            lw=2.5, ms=8, label="optimised designs (guaranteed strength)")
    ax.set(title=f"Optimised {lab} vs strength", xlabel="28-day strength [MPa]", ylabel=lab)
    ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "12_design_frontier")
plt.show()

feas = designs[designs["feasible"]]
if len(feas):
    c40 = feas[(feas["objective"] == "co2")].sort_values("target MPa")
    scm_co2, scm_cost = c40["SCM %"].mean(), feas[feas["objective"] == "cost"]["SCM %"].mean()
    learn("concrete", f"The optimiser found feasible designs for {len(feas)} of {len(designs)} target/objective combinations. "
          f"Low-CO₂ designs use {scm_co2:.0f}% SCM on average at w/b ≈ {c40['w/b'].mean():.2f}, "
          f"compared with {scm_cost:.0f}% SCM in the cheapest designs. "
          + ("The model reduces emissions mainly by replacing cement with slag or fly ash while keeping w/b low."
             if scm_co2 > scm_cost + 5 else "The cheapest and the lowest-CO₂ designs are similar, because both objectives favour low cement content."))

---
## 13 · Key Learnings

### 13.1 Findings from this run

In [ ]:
TITLES = {"concrete": "What the data teaches about concrete", "data": "What we learn about the dataset",
          "modelling": "What we learn about modelling", "process": "What we learn about the analysis process"}
for cat in ["concrete", "data", "modelling", "process"]:
    if LEARNINGS.get(cat):
        display(Markdown(f"#### {TITLES[cat]}\n" + "\n".join(f"{i}. {t}" for i, t in enumerate(LEARNINGS[cat], 1))))

### 13.2 Methodological lessons from revisiting the original project
1. **Split by mix, not by row.** A mix tested at several ages is a single design. Random row splits let the model recognise the design and interpolate across age, which inflates accuracy for the real use case of predicting a new mix.
2. **Encode domain knowledge as features.** Water/binder ratio, SCM fraction and log-age condense a century of concrete research. They make simple models competitive and also make complex models easier to interpret.
3. **Keep a physical benchmark.** A five-parameter Abrams-and-age model sets the level of accuracy that a black-box model has to justify, and it remains usable when the ML model is not trusted.
4. **Report uncertainty, not only accuracy.** For structural use, what matters is the strength that can be guaranteed. Prediction intervals turn a prediction into a design value.
5. **Optimise only inside the data.** An optimiser will exploit every weakness of a model. Constraints on bounds, volume and the applicability domain keep its recommendations realistic.
6. **Interpret with care.** Impurity-based importances and single correlations mix age with composition. Composition-aware permutation and ICE curves give answers that can be checked against engineering knowledge.

### 13.3 Limits of the data
- The data are laboratory specimens from one source compilation. Field curing, cement type, aggregate mineralogy, admixture chemistry and air content are not recorded.
- Ages beyond 100 days and very high-strength mixes are sparsely represented, so predictions there are less reliable.
- Prices and CO₂ factors in the optimiser are illustrative and must be replaced with local values.

In [ ]:
joblib.dump({"model": tuned[CHAMPION], "physics_model": tuned["PHYS"], "inputs": RAW_INPUTS, "interval_halfwidth": Q_HALF,
             "domain": {"scaler": dom_scaler, "nn": dom_nn, "limit": DOMAIN_LIMIT}}, DIRS["model"] / "concrete_strength_model.joblib")
card = {"created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "champion": MODEL_NAMES[CHAMPION],
        "cv_rmse": float(tuning["tuned CV RMSE"].iloc[0]), "test_rmse": float(ch["RMSE"]), "interval_halfwidth": Q_HALF,
        "physics_params": dict(zip(["A", "k", "a", "b", "c"], map(float, tuned["PHYS"].params_))), "versions": VERSIONS}
(DIRS["model"] / "model_card.json").write_text(json.dumps(card, indent=2))
archive = shutil.make_archive(str(HOME), "zip", HOME)
print(f"Archive: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(archive)

---
**References.** Yeh, I-C. (1998). *Modeling of strength of high-performance concrete using artificial neural networks.* Cement and Concrete Research 28(12), 1797–1808 · Abrams, D. A. (1918). *Design of Concrete Mixtures.* Lewis Institute Bulletin 1 · ACI 209R-92, *Prediction of Creep, Shrinkage, and Temperature Effects in Concrete Structures* · Storn, R. & Price, K. (1997). *Differential evolution*, J. Global Optimization · Vovk, Gammerman & Shafer (2005), *Algorithmic Learning in a Random World*.